# 05 — PlantVillage Segmentation Model Comparison

Final comparison of:

- U-Net + ResNet34
- DeepLabV3+ + ResNet34
- FPN + ResNet34

This notebook is **CPU-only**; no GPU is required.

Before running, place the lightweight experiment folders from all three runs into the **same Google Drive**:

```text
MyDrive/PlantVillage_Segmentation/experiments/
├── UNet/
│   ├── unet_config.json
│   └── results/
├── DeepLabV3Plus/
│   ├── deeplabv3plus_config.json
│   └── results/
└── FPN/
    ├── fpn_config.json
    └── results/
```

The `.pth` checkpoint files are **not required** for this comparison notebook.


# STEP 1 — Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# STEP 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/PlantVillage_Segmentation"
)

EXPERIMENTS_DIR = PROJECT_DIR / "experiments"

print(EXPERIMENTS_DIR)
assert EXPERIMENTS_DIR.exists()


# STEP 3 — Define Experiment Paths

In [ ]:
MODEL_PATHS = {
    "U-Net": {
        "root": EXPERIMENTS_DIR / "UNet",
        "summary": EXPERIMENTS_DIR / "UNet/results/experiment_summary.json",
        "metrics": EXPERIMENTS_DIR / "UNet/results/test_metrics.json",
        "history": EXPERIMENTS_DIR / "UNet/results/training_history.csv",
        "config": EXPERIMENTS_DIR / "UNet/unet_config.json",
    },
    "DeepLabV3+": {
        "root": EXPERIMENTS_DIR / "DeepLabV3Plus",
        "summary": EXPERIMENTS_DIR / "DeepLabV3Plus/results/experiment_summary.json",
        "metrics": EXPERIMENTS_DIR / "DeepLabV3Plus/results/test_metrics.json",
        "history": EXPERIMENTS_DIR / "DeepLabV3Plus/results/training_history.csv",
        "config": EXPERIMENTS_DIR / "DeepLabV3Plus/deeplabv3plus_config.json",
    },
    "FPN": {
        "root": EXPERIMENTS_DIR / "FPN",
        "summary": EXPERIMENTS_DIR / "FPN/results/experiment_summary.json",
        "metrics": EXPERIMENTS_DIR / "FPN/results/test_metrics.json",
        "history": EXPERIMENTS_DIR / "FPN/results/training_history.csv",
        "config": EXPERIMENTS_DIR / "FPN/fpn_config.json",
    },
}

for model_name, paths in MODEL_PATHS.items():
    print("\n", model_name)
    for key in ["summary", "metrics", "history", "config"]:
        print(
            "✅" if paths[key].exists() else "❌",
            key,
            paths[key]
        )
        assert paths[key].exists(), f"Missing {key} for {model_name}"


# STEP 4 — Load All Experiment Results

In [ ]:
summaries = {}
metrics = {}
histories = {}
configs = {}

for model_name, paths in MODEL_PATHS.items():

    with open(paths["summary"], "r") as f:
        summaries[model_name] = json.load(f)

    with open(paths["metrics"], "r") as f:
        metrics[model_name] = json.load(f)

    histories[model_name] = pd.read_csv(
        paths["history"]
    )

    with open(paths["config"], "r") as f:
        configs[model_name] = json.load(f)

print("✅ All experiment files loaded.")


# STEP 5 — Build Final Comparison Table

In [ ]:
rows = []

for model_name in MODEL_PATHS:

    summary = summaries[model_name]
    test = metrics[model_name]
    config = configs[model_name]

    rows.append({
        "Model": model_name,
        "Encoder": summary.get("encoder", config.get("encoder", "resnet34")),
        "Batch Size": config.get("batch_size"),
        "Epochs Completed": summary.get("epochs_completed"),
        "Best Epoch": summary.get("best_epoch"),
        "Best Val Dice": summary.get("best_validation_dice"),
        "Best Val IoU": summary.get("best_validation_iou"),
        "Test Loss": test.get("test_loss"),
        "Test Dice": test.get("dice"),
        "Test IoU": test.get("iou"),
        "Precision": test.get("precision"),
        "Recall": test.get("recall"),
        "Specificity": test.get("specificity"),
        "Pixel Accuracy": test.get("pixel_accuracy"),
    })

comparison_df = pd.DataFrame(rows)

display(
    comparison_df.style.format({
        "Best Val Dice": "{:.6f}",
        "Best Val IoU": "{:.6f}",
        "Test Loss": "{:.6f}",
        "Test Dice": "{:.6f}",
        "Test IoU": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "Specificity": "{:.6f}",
        "Pixel Accuracy": "{:.6f}",
    })
)


# STEP 6 — Rank Models

In [ ]:
ranking_df = comparison_df[
    [
        "Model",
        "Test Dice",
        "Test IoU",
        "Precision",
        "Recall",
        "Specificity",
        "Pixel Accuracy"
    ]
].copy()

ranking_df["Dice Rank"] = ranking_df[
    "Test Dice"
].rank(
    ascending=False,
    method="min"
).astype(int)

ranking_df["IoU Rank"] = ranking_df[
    "Test IoU"
].rank(
    ascending=False,
    method="min"
).astype(int)

ranking_df = ranking_df.sort_values(
    ["Dice Rank", "IoU Rank"]
).reset_index(drop=True)

display(ranking_df)

best_model = ranking_df.iloc[0]["Model"]
print("\nBest observed model by Test Dice:", best_model)


# STEP 7 — Compute Differences from the Best Model

In [ ]:
best_dice = comparison_df["Test Dice"].max()
best_iou = comparison_df["Test IoU"].max()

delta_df = comparison_df[
    ["Model", "Test Dice", "Test IoU"]
].copy()

delta_df["Dice Gap vs Best"] = (
    best_dice - delta_df["Test Dice"]
)

delta_df["IoU Gap vs Best"] = (
    best_iou - delta_df["Test IoU"]
)

display(
    delta_df.style.format({
        "Test Dice": "{:.6f}",
        "Test IoU": "{:.6f}",
        "Dice Gap vs Best": "{:.6f}",
        "IoU Gap vs Best": "{:.6f}",
    })
)


# STEP 8 — Plot Test Dice and IoU Comparison

In [ ]:
plot_df = comparison_df.set_index("Model")

ax = plot_df[
    ["Test Dice", "Test IoU"]
].plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_ylabel("Score")
ax.set_title("Segmentation Model Comparison: Dice and IoU")
ax.set_ylim(0.97, 1.0)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()

COMPARISON_DIR = (
    PROJECT_DIR /
    "experiments" /
    "Model_Comparison"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

dice_iou_path = (
    COMPARISON_DIR /
    "dice_iou_comparison.png"
)

plt.savefig(
    dice_iou_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", dice_iou_path)


# STEP 9 — Plot Precision / Recall / Specificity / Accuracy

In [ ]:
ax = plot_df[
    [
        "Precision",
        "Recall",
        "Specificity",
        "Pixel Accuracy"
    ]
].plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_ylabel("Score")
ax.set_title("Segmentation Model Comparison: Pixel-Level Metrics")
ax.set_ylim(0.985, 1.0)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()

metric_path = (
    COMPARISON_DIR /
    "pixel_metrics_comparison.png"
)

plt.savefig(
    metric_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", metric_path)


# STEP 10 — Compare Validation Dice Across Epochs

In [ ]:
plt.figure(figsize=(9, 5))

for model_name, history in histories.items():

    plt.plot(
        history["epoch"],
        history["val_dice"],
        label=model_name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Dice")
plt.title("Validation Dice Across Training")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

history_plot_path = (
    COMPARISON_DIR /
    "validation_dice_training_comparison.png"
)

plt.savefig(
    history_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", history_plot_path)

print(
    "\nNote: training budgets were not identical across all three models; "
    "this plot is descriptive rather than a strictly controlled convergence test."
)


# STEP 11 — Save Final Comparison CSV / JSON

In [ ]:
COMPARISON_CSV = (
    COMPARISON_DIR /
    "segmentation_model_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_CSV,
    index=False
)

comparison_json = {
    "best_model_by_test_dice": best_model,
    "models": comparison_df.to_dict(
        orient="records"
    ),
    "methodological_note": (
        "All models used the same final dataset, split, target rule, "
        "ResNet34 ImageNet encoder family, loss family, and final test protocol. "
        "DeepLabV3+ used a different batch size and shorter stopping budget, "
        "so very small architecture differences should be interpreted cautiously."
    )
}

COMPARISON_JSON = (
    COMPARISON_DIR /
    "segmentation_model_comparison.json"
)

with open(
    COMPARISON_JSON,
    "w"
) as f:
    json.dump(
        comparison_json,
        f,
        indent=4
    )

print("Saved:", COMPARISON_CSV)
print("Saved:", COMPARISON_JSON)


# STEP 12 — Generate Paper-Ready LaTeX Results Table

In [ ]:
latex_table = comparison_df[
    [
        "Model",
        "Test Dice",
        "Test IoU",
        "Precision",
        "Recall",
        "Specificity",
        "Pixel Accuracy"
    ]
].copy()

latex_text = latex_table.to_latex(
    index=False,
    float_format=lambda x: f"{x:.6f}",
    caption="Test-set segmentation performance on the final PlantVillage segmentation split.",
    label="tab:segmentation_comparison"
)

LATEX_TABLE_PATH = (
    COMPARISON_DIR /
    "segmentation_comparison_table.tex"
)

LATEX_TABLE_PATH.write_text(
    latex_text,
    encoding="utf-8"
)

print(latex_text)
print("\nSaved:", LATEX_TABLE_PATH)


# STEP 13 — Print Final Interpretation

In [ ]:
sorted_models = comparison_df.sort_values(
    "Test Dice",
    ascending=False
).reset_index(drop=True)

first = sorted_models.iloc[0]
second = sorted_models.iloc[1]
third = sorted_models.iloc[2]

print("=" * 70)
print("FINAL SEGMENTATION COMPARISON")
print("=" * 70)

print(
    f"1st: {first['Model']} | "
    f"Dice={first['Test Dice']:.6f} | "
    f"IoU={first['Test IoU']:.6f}"
)

print(
    f"2nd: {second['Model']} | "
    f"Dice={second['Test Dice']:.6f} | "
    f"IoU={second['Test IoU']:.6f}"
)

print(
    f"3rd: {third['Model']} | "
    f"Dice={third['Test Dice']:.6f} | "
    f"IoU={third['Test IoU']:.6f}"
)

print(
    "\nImportant: the numerical gaps are very small. "
    "Report the observed ranking, but do not overclaim universal model superiority."
)


# GitHub Files from This Notebook

After the comparison finishes, copy these lightweight outputs into:

```text
results/segmentation/comparison/
```

Recommended files:

- `segmentation_model_comparison.csv`
- `segmentation_model_comparison.json`
- `dice_iou_comparison.png`
- `pixel_metrics_comparison.png`
- `validation_dice_training_comparison.png`
- `segmentation_comparison_table.tex`

No GPU checkpoints are required.
